# 임시 진단: Ollama · LangChain · Structured Output 호출 분리

이 노트북은 14번의 `하닉` 사전 호출이 오래 걸린 원인을 분리하기 위한 **임시 진단용** 파일이다. 성능 비교·최종 구조 선택·결과 보고서 작성 목적이 아니며, 진단 후 삭제할 수 있다.

> 현재 사용자가 GPU에 올린 Ollama runner를 그대로 사용한다. `num_ctx`, GPU, keep-alive, 출력 길이 같은 runner 옵션은 지정하지 않는다.

## 실행 순서

아래 네 호출은 **각각 별도 셀**이다. 앞 셀이 정상 완료된 경우에만 다음 셀을 실행한다. 어느 셀에서 오래 멈추는지와 마지막 출력 문구를 기록한다.

1. Ollama 직접 chat 호출: 모델·Ollama 기본 경로
2. LangChain `ChatOllama` 일반 호출: LangChain 연결 경로
3. LangChain structured output 호출: 일반 구조화 출력 경로
4. 14번 사전 호출 동일 조건 재현: 14번의 정확한 프롬프트·스키마·클라이언트 경로

`temperature=0`은 14번과 동일한 생성 조건을 맞추기 위한 요청 단위 옵션이며, runner 적재 설정은 바꾸지 않는다. HTTP 대기 제한은 120초다. timeout이 나면 다음 셀을 실행하지 않고, Ollama 요청이 남아 있는지 먼저 확인한다.

In [1]:
import json
import os
from pathlib import Path
import time
from typing import Literal
import urllib.request

from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
MODEL_NAME = os.environ.get('OLLAMA_MODEL', 'qwen3.6:27b')
TEMPERATURE = 0

print(f'모델: {MODEL_NAME}')
print(f'Ollama URL: {OLLAMA_BASE_URL}')
print(f'temperature: {TEMPERATURE}')
print('runner 옵션: 지정하지 않음')

DIAGNOSTIC_MESSAGES = [
    ('system', '짧고 정확하게 답하세요.'),
    ('human', '하닉은 어떤 회사의 약칭으로 해석할 수 있나요?'),
]

모델: qwen3.6:27b
Ollama URL: http://localhost:11434
temperature: 0
runner 옵션: 지정하지 않음


## 1. Ollama 직접 chat 호출

LangChain 없이 Ollama `/api/chat`을 직접 호출한다. 2번과 동일한 메시지를 사용한다. 이 셀이 오래 멈추면 LangChain 이전의 모델·Ollama 실행 경로부터 확인해야 한다.

In [3]:
payload = {
    'model': MODEL_NAME,
    'messages': [
        {'role': 'user' if role == 'human' else role, 'content': content}
        for role, content in DIAGNOSTIC_MESSAGES
    ],
    'stream': False,
    'options': {'temperature': TEMPERATURE},
}
request = urllib.request.Request(
    f'{OLLAMA_BASE_URL}/api/chat',
    data=json.dumps(payload).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
)

print('[1/4] Ollama 직접 chat 호출 시작', flush=True)
started_at = time.perf_counter()
try:
    with urllib.request.urlopen(request, timeout=120) as response:
        direct_result = json.load(response)
    print(f'[1/4] 완료 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print('content:', repr(direct_result.get('message', {}).get('content')))
    print('thinking:', repr(direct_result.get('message', {}).get('thinking')))
    print('생성 토큰 수:', direct_result.get('eval_count'))
except Exception as error:
    print(f'[1/4] 실패 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print(f'{type(error).__name__}: {error}')

[1/3] Ollama 직접 chat 호출 시작
[1/3] 완료 / 51.29초
content: '네, 알겠습니다. 질문해 주세요.'
thinking: 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "짧고 정확하게 답하세요." (Korean for "Answer briefly and accurately.")\n   - This is a directive/instruction, not a specific question.\n   - I need to acknowledge the instruction and confirm I\'ll follow it, while keeping my response short and precise as requested.\n\n2.  **Identify Key Requirements:**\n   - Language: Korean\n   - Tone: Professional, concise\n   - Content: Acknowledge the instruction, confirm compliance, keep it brief.\n\n3.  **Formulate Response (Internal Refinement):**\n   - Option 1: "네, 알겠습니다. 질문해 주세요." (Yes, understood. Please ask your question.)\n   - Option 2: "네, 짧고 정확하게 답변드리겠습니다. 무엇을 도와드릴까요?" (Yes, I\'ll answer briefly and accurately. How can I help?)\n   - Option 3: "네, 알겠습니다. 질문을 입력해 주세요." (Yes, understood. Please enter your question.)\n   - All are short. Option 1 is the most direct and matches the "brief a

## 2. LangChain `ChatOllama` 일반 호출

동일 모델을 LangChain으로 부르되 structured output은 사용하지 않는다. 1번은 빠르고 이 셀만 오래 멈추면 LangChain 연결·메시지 변환 경로를 우선 의심한다.

In [4]:
llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=TEMPERATURE,
    client_kwargs={'timeout': 120},
)

print('[2/4] LangChain 일반 호출 시작', flush=True)
started_at = time.perf_counter()
try:
    plain_response = llm.invoke(DIAGNOSTIC_MESSAGES)
    print(f'[2/4] 완료 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print('content:', repr(plain_response.content))
    print('thinking:', repr(plain_response.additional_kwargs.get('thinking')))
except Exception as error:
    print(f'[2/4] 실패 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print(f'{type(error).__name__}: {error}')

[2/3] LangChain 일반 호출 시작
[2/3] 완료 / 197.62초
content: '하닉은 일반적으로 특정 회사의 공식 약칭으로 사용되지 않습니다. 유사한 발음의 주요 기업은 다음과 같습니다:\n\n- **SK하이닉스**(SK Hynix): 반도체 메모리 전문 기업\n- **한컴**(Hancom): 소프트웨어 및 교육 서비스 기업\n- **한일그룹**(Hannil Group): 건설 및 무역 기업\n\n의도하신 회사가 있다면 정확한 명칭을 알려주세요.'
thinking: None


## 3. LangChain structured output 호출

14번의 사전 호출과 같은 형식이다. 이 셀만 오래 멈추면, 모델 자체보다 structured output 형식 또는 해당 프롬프트와의 상호작용을 원인 후보로 좁힌다.

In [5]:
class EntitySearchPlan(BaseModel):
    decision: Literal['candidate', 'ambiguous', 'unknown'] = Field(
        description='검색 가설 제안, 포괄·다의적 표현, 또는 해석 불가 상태'
    )
    search_terms: list[str] = Field(
        default_factory=list,
        max_length=2,
        description='OpenDART에서 검증할 정식 기업명 검색 가설'
    )

SYSTEM_PROMPT = '''
당신은 한국 기업 리서치 시스템의 기업명 검색 계획기다.
원문이 명백한 약칭, 한글·영문 표기 차이, 또는 가벼운 철자 오타로 보이면 decision='candidate'와 함께
정식 기업명으로 보이는 검색 가설을 최대 2개 제안한다.
원문이 포괄적인 그룹명, 여러 기업을 뜻할 수 있는 표현, 또는 실제 기업으로 해석할 근거가 없는 표현이면
decision='ambiguous' 또는 decision='unknown'을 반환하고 search_terms=[]로 둔다.
corp_code, 종목코드, 투자 의견, 설명 문장을 반환하지 않는다.
'''

structured_llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=TEMPERATURE,
    client_kwargs={'timeout': 120},
)
planner = structured_llm.with_structured_output(EntitySearchPlan, include_raw=True)

print('[3/4] LangChain structured output 호출 시작', flush=True)
started_at = time.perf_counter()
try:
    structured_result = planner.invoke([
        ('system', SYSTEM_PROMPT),
        ('human', '사용자 질문: 하닉 실적을 정리해줘.\n원문 기업 표현: 하닉'),
    ])
    print(f'[3/4] 완료 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print('파싱 결과:', structured_result['parsed'])
    print('파싱 오류:', structured_result['parsing_error'])
    print('원문 응답:', structured_result['raw'])
except Exception as error:
    print(f'[3/4] 실패 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print(f'{type(error).__name__}: {error}')

[3/3] LangChain structured output 호출 시작
[3/3] 완료 / 69.81초
파싱 결과: decision='candidate' search_terms=['하닉홀딩스', '하닉시스템']
파싱 오류: None
원문 응답: content='{\n  "decision": "candidate",\n  "search_terms": ["하닉홀딩스", "하닉시스템"]\n}' additional_kwargs={} response_metadata={'model': 'qwen3.6:27b', 'created_at': '2026-09-14T12:25:50.919388416Z', 'done': True, 'done_reason': 'stop', 'total_duration': 69794577231, 'load_duration': 194968007, 'prompt_eval_count': 2302, 'prompt_eval_duration': 538935570, 'eval_count': 30, 'eval_duration': 929309673, 'logprobs': None, 'model_name': 'qwen3.6:27b', 'model_provider': 'ollama'} id='lc_run--01a09fe0-6fa2-7173-8137-b69ec2087d15-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 2302, 'output_tokens': 30, 'total_tokens': 2332}


## 4. 14번 사전 호출 동일 조건 재현

3번이 정상 종료됐어도 14번과 시스템 프롬프트·스키마 세부 설명·클라이언트 생성 방식이 다르면 장시간 호출을 설명할 수 없다. 이 셀은 14번의 사전 호출 코드를 동일하게 재현한다.

> 이 셀에는 `client_kwargs`, `num_ctx`, GPU, keep-alive, 생성 길이 옵션을 넣지 않는다. 14번과 마찬가지로 모델·URL·`temperature=0`만 지정한다. 다시 장시간 멈추면 커널에서 중단하고, 이 셀 이후로 진행하지 않는다.

In [ ]:
# 14번과 동일한 Pydantic 클래스명으로 다시 정의한다. 클래스명도 JSON schema에 포함될 수 있다.
class EntitySearchPlan(BaseModel):
    decision: Literal['candidate', 'ambiguous', 'unknown'] = Field(
        description='검색 가설 제안, 포괄·다의적 표현, 또는 해석 불가 상태'
    )
    search_terms: list[str] = Field(
        default_factory=list,
        max_length=2,
        description='OpenDART에서 검증할 정식 기업명 검색 가설. corp_code·종목코드·설명 문장은 금지.'
    )

V14_SYSTEM_PROMPT = '''
당신은 한국 기업 리서치 시스템의 기업명 검색 계획기다.
프로그램은 원문 기업 표현을 OpenDART에서 고유하게 exact 확정하지 못했을 때만 당신을 호출한다.

원문이 명백한 약칭, 한글·영문 표기 차이, 또는 가벼운 철자 오타로 보이면 decision='candidate'와 함께
정식 기업명으로 보이는 검색 가설을 최대 2개 제안한다. 완전한 확신이 없어도 검색 가설은 제안할 수 있다.
이 가설은 정답 확정이 아니며, 뒤의 프로그램이 OpenDART 기업 목록으로 검증하고 필요하면 사용자에게 재질문한다.

원문이 포괄적인 그룹명, 여러 기업을 뜻할 수 있는 표현, 또는 실제 기업으로 해석할 근거가 없는 표현이면
decision='ambiguous' 또는 decision='unknown'을 반환하고 search_terms=[]로 둔다.

사용자가 언급하지 않은 계열사·경쟁사·관련 기업을 추가하지 않는다.
corp_code, 종목코드, 투자 의견, 설명 문장을 반환하지 않는다.
'''

v14_llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=TEMPERATURE,
)
v14_planner = v14_llm.with_structured_output(EntitySearchPlan, include_raw=True)

print('[4/4] 14번 사전 호출 동일 조건 재현 시작 (앞선 호출로 warm-up된 상태)', flush=True)
started_at = time.perf_counter()
try:
    v14_result = v14_planner.invoke([
        ('system', V14_SYSTEM_PROMPT),
        ('human', '사용자 질문: 하닉 실적을 정리해줘.\n원문 기업 표현: 하닉'),
    ])
    print(f'[4/4] 완료 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print('파싱 결과:', v14_result['parsed'])
    print('파싱 오류:', v14_result['parsing_error'])
    print('원문 응답:', v14_result['raw'])
except Exception as error:
    print(f'[4/4] 실패 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print(f'{type(error).__name__}: {error}')

## 해석 기준

- 1번부터 오래 멈춤: Ollama·모델 실행 경로 또는 현재 runner 상태가 원인 후보
- 1번 정상, 2번부터 오래 멈춤: LangChain `ChatOllama` 연결·메시지 경로가 원인 후보
- 1·2번 정상, 3번만 오래 멈춤: structured output 형식·프롬프트 상호작용이 원인 후보
- 1·2·3번 정상, 4번만 장시간 멈춤: 14번의 상세 시스템 프롬프트·스키마 설명 또는 client 설정 차이가 원인 후보
- 1·2·3·4번 모두 정상: 현재의 warm-up된 조건에서는 재현되지 않은 것이다. 당시 14번 실행의 일시적 runner 상태·GPU 경합·커널 상태가 원인 후보이지만, 이 결과만으로 원인을 확정하지 않는다.

이 결과는 원인 후보를 좁히는 용도다. 단일 실행만으로 최종 모델·프롬프트·구조를 채택하거나 미채택하지 않는다.